# Module 4: Search in Azure DocumentDB

**Time**: ~75 min  
**Environment**: Jupyter notebook in VS Code

Before starting, open a PowerShell terminal in this notebook's folder and run `az login`, followed by `../../../1-DocumentDB-Introduction-and-Cluster-Setup/Set-LabEnvironment.ps1`. Restart the notebook kernel so it reads the configured environment variables. Run each cell in order.

The notebook creates embeddings for the sample documents, stores them in Azure DocumentDB, creates vector and full-text indexes, and runs vector, BM25, fuzzy, phrase, and hybrid search.

> Full-text search in Azure DocumentDB is currently in gated preview. DiskANN vector search requires an M30 or higher cluster tier.

## Step 0: Connect and configure embeddings

The required Python packages are provided on the workshop VM. This cell reads the DocumentDB cluster name, Azure OpenAI endpoint, and embedding deployment name, then creates both clients. Both services authenticate with your active Azure CLI session through `AzureCliCredential`; run `az login` first.


In [ ]:
import os

from azure.identity import AzureCliCredential, get_bearer_token_provider
from pymongo import MongoClient
from pymongo.auth_oidc import OIDCCallback, OIDCCallbackContext, OIDCCallbackResult
from pymongo.errors import OperationFailure
from openai import OpenAI


def require_env(name):
    value = os.environ.get(name)
    if not value:
        raise RuntimeError(
            f"{name} is not set. Run ../../../1-DocumentDB-Introduction-and-Cluster-Setup/"
            "Set-LabEnvironment.ps1, then restart the notebook kernel."
        )
    return value


class AzureIdentityTokenCallback(OIDCCallback):
    def __init__(self, credential):
        self.credential = credential

    def fetch(self, context: OIDCCallbackContext) -> OIDCCallbackResult:
        del context
        token = self.credential.get_token("https://ossrdbms-aad.database.windows.net/.default")
        return OIDCCallbackResult(access_token=token.token)


documentdb_connection_uri = require_env("DOCUMENTDB_CONNECTION_URI")
cluster_name = require_env("DOCUMENTDB_CLUSTER_NAME")
azure_openai_endpoint = require_env("AZURE_OPENAI_ENDPOINT")
embedding_model = require_env("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")

credential = AzureCliCredential()
auth_properties = {"OIDC_CALLBACK": AzureIdentityTokenCallback(credential)}
client = MongoClient(
    documentdb_connection_uri,
    tls=True,
    retryWrites=False,
    authMechanism="MONGODB-OIDC",
    authMechanismProperties=auth_properties,
)
db = client["docdbworkshop"]
collection = db["workshop_content"]
openai_token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")
openai_client = OpenAI(
    base_url=f"{azure_openai_endpoint.rstrip('/')}/openai/v1/",
    api_key=openai_token_provider,
)
print(db.command({"ping": 1}))
print("DocumentDB cluster:", cluster_name)
print("Embedding deployment:", embedding_model)

## Step 1: Generate embeddings and load sample documents

This cell calls the OpenAI embeddings API for each sample document body, stores the resulting vector in the `embedding` field, and inserts the documents into Azure DocumentDB.

In [ ]:
def create_embedding(text: str) -> list[float]:
    response = openai_client.embeddings.create(model=embedding_model, input=text)
    return response.data[0].embedding

source_docs = [
    {"_id":"doc-search-001","title":"DiskANN vector indexing","category":"vector","body":"Azure DocumentDB supports DiskANN vector indexes for high recall semantic similarity search over embeddings stored with documents.","sku":"SEARCH-VEC-001"},
    {"_id":"doc-search-002","title":"BM25 keyword search","category":"full-text","body":"Azure DocumentDB full-text search ranks keyword matches with BM25 and exposes scores through searchScore metadata.","sku":"SEARCH-FTS-001"},
    {"_id":"doc-search-003","title":"Hybrid search with RRF","category":"hybrid","body":"Hybrid search combines BM25 keyword results with vector results and fuses the ranked lists using Reciprocal Rank Fusion.","sku":"SEARCH-HYB-001"},
    {"_id":"doc-search-004","title":"RAG grounding","category":"rag","body":"Retrieval augmented generation retrieves relevant chunks from Azure DocumentDB and grounds the model answer in that context.","sku":"RAG-PIPE-001"},
    {"_id":"doc-search-005","title":"Operational filtering","category":"filters","body":"Search applications often filter by status, tenant, region, stock, or category after the search stage narrows candidate documents.","sku":"SEARCH-FLT-001"}
]

collection.drop()
for doc in source_docs:
    doc["embedding"] = create_embedding(doc["body"])
collection.insert_many(source_docs)
embedding_dimensions = len(source_docs[0]["embedding"])
print("Loaded documents:", collection.count_documents({}))
print("Embedding dimensions:", embedding_dimensions)

## Step 2: Create vector and full-text indexes

The vector index uses `cosmosSearch` and the actual embedding dimension returned by Azure OpenAI. The full-text index uses `createSearchIndexes` over the `body` field.

**STUDENT EXERCISE:** create both indexes in the next cell. Wrap full-text index creation in `try`/`except OperationFailure`. Set `full_text_search_supported` to `False` and continue when `error.code == 115`; re-raise every other error. Compare with the matching Python `after` notebook if you get stuck.


In [ ]:
# STUDENT EXERCISE: create both indexes.
# 1. Vector: createIndexes on workshop_content with key {"embedding": "cosmosSearch"}.
# 2. Full-text: createSearchIndexes named idx_body_fts over body.
# Use embedding_dimensions for vector dimensions.
# Catch only OperationFailure code 115 and set full_text_search_supported = False.

full_text_search_supported = False
vector_index_result = {"ok": 0, "message": "TODO: create idx_embedding_diskann"}
full_text_index_result = {"ok": 0, "message": "TODO: create idx_body_fts with code-115 handling"}
{"vector": vector_index_result, "fullText": full_text_index_result}


## Step 3: Generate a query embedding and run vector search

The query text is embedded with the same model, then used as the `vector` in `$search.cosmosSearch`.

**STUDENT EXERCISE:** complete the vector retrieval query. Expected result: top chunks/documents should relate to RAG, vector, or hybrid search.


In [ ]:
search_text = "semantic retrieval for RAG"
query_vector = create_embedding(search_text)

vector_pipeline = [
    {"$search": {"cosmosSearch": {
        "path": "embedding",
        "vector": query_vector,
        "k": 3,
        "lSearch": 40,
    }}},
    {"$project": {
        "_id": 0,
        "title": 1,
        "category": 1,
        "score": {"$meta": "searchScore"},
    }},
]

list(collection.aggregate(vector_pipeline))

## Step 4: Run BM25 full-text search

This query uses `$search.text` against the named full-text index and projects BM25 relevance scores with `$meta: "searchScore"`.

**STUDENT EXERCISE:** complete the search or hybrid retrieval snippet in the next cell. Notice where `$limit` belongs and how `searchScore` is projected.


In [ ]:
if full_text_search_supported:
    bm25_results = list(collection.aggregate([
        {"$search": {"index": "idx_body_fts", "text": {"query": "BM25 ranking", "path": "body"}}},
        {"$limit": 5},
        {"$project": {"_id": 0, "title": 1, "body": 1, "score": {"$meta": "searchScore"}}}
    ]))
else:
    bm25_results = []
    print("Skipped BM25 search because full-text search is not enabled.")
bm25_results


## Step 5: Run fuzzy and phrase search

Fuzzy search tolerates typos with `maxEdits`. Phrase search requires ordered terms, with `slop` controlling how close the words must be.

In [ ]:
if full_text_search_supported:
    fuzzy_results = list(collection.aggregate([
        {"$search": {"index": "idx_body_fts", "text": {"query": "retrival augmentd genration", "path": "body", "fuzzy": {"maxEdits": 1}}}},
        {"$limit": 5},
        {"$project": {"_id": 0, "title": 1, "score": {"$meta": "searchScore"}}}
    ]))
    phrase_results = list(collection.aggregate([
        {"$search": {"index": "idx_body_fts", "phrase": {"query": "Reciprocal Rank Fusion", "path": "body", "slop": 0}}},
        {"$limit": 5},
        {"$project": {"_id": 0, "title": 1, "score": {"$meta": "searchScore"}}}
    ]))
else:
    fuzzy_results, phrase_results = [], []
    print("Skipped fuzzy and phrase search because full-text search is not enabled.")
{"fuzzy": fuzzy_results, "phrase": phrase_results}


## Step 6: Run hybrid search

Hybrid search embeds the user query, runs vector and BM25 retrieval, and fuses both ranked lists with RRF.

In [ ]:
search_text = "semantic retrieval for RAG"
query_vector = create_embedding(search_text)

vector_hits = list(collection.aggregate([
    {"$search": {"cosmosSearch": {
        "path": "embedding",
        "vector": query_vector,
        "k": 5,
        "lSearch": 40,
    }}},
    {"$project": {"_id": 1, "title": 1}},
]))

if full_text_search_supported:
    keyword_hits = list(collection.aggregate([
        {"$search": {"index": "idx_body_fts", "text": {
            "query": search_text,
            "path": "body",
        }}},
        {"$limit": 5},
        {"$project": {"_id": 1, "title": 1}},
    ]))

    scores = {}
    documents = {}
    for results in (keyword_hits, vector_hits):
        for rank, document in enumerate(results):
            document_id = str(document["_id"])
            documents[document_id] = document
            scores[document_id] = scores.get(document_id, 0) + 1 / (60 + rank + 1)

    combined_results = [
        {**documents[document_id], "rrfScore": score}
        for document_id, score in sorted(scores.items(), key=lambda item: item[1], reverse=True)[:5]
    ]
else:
    keyword_hits = []
    combined_results = vector_hits
    print("Full-text search is unavailable; showing vector results instead of RRF.")
combined_results
